## 2. 数据集准备

评测 RAG 应用，数据集必须有：

- 运行输入：
    - question[str]：问题
- 运行输出
    - answer[str]：RAG 应用给出的回答
    - contexts[list[str]]: 检索到的上下文，顺序则代表相似度。
- 评估输入
    - reference_context[str]: 参考上下文，用于评估检索的正确性。
    - reference_answer[str]: 参考答案，用于评估回答的正确性。

数据集有三种方法准备：

1. 使用开源数据集，比如 ragas 引用的 explodinggradients/fiqa 数据集。但是中文的 RAG 数据集较少。而且开源数据集只能代表 RAG 的通用能力，不能代表 RAG 在特定领域的能力。
2. 人工标注，这种方法需要大量的人力成本，但是可以标注特定领域的数据集。
3. 使用 LLM 自动抽取 QA 对，从而形成数据集。这种方法的优点是成本低，缺点是数据集的质量可能不高。自动标注尽量使用能力较强的 LLM，比如 GPT-4 等。

### 2.1 使用开源数据集

#### 2.1.1 cmrc 数据集下载和转换

正式项目中，应该以全部的数据作为我们的向量索引的基础数据，单条数据格式如上。我们挑选其中的 context 嵌入到向量索引中。

本次我们就只以 test 的 1000 条数据建立索引。同时以 test 的随机 50 条查询作为验证数据集。

此时我们可以准备两种数据集

1. 用于运行的数据集，即只有 question / reference_answer / reference_contexts 的数据集。
2. 用于评估的数据集，即使用 chain 检索后的数据集，包含全部数据。

前者在启动评估任务的时候还要注册 Provider 用于 RAG 生成，为模拟真实的情况，我们选择 1。那么此处只需要生成用于运行的数据集即可。


In [7]:
import json
import os
import pandas as pd

data_dir = 'cmrcData/data'
cmrc = {}
cmrc['train'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_train.json'))
cmrc['test'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_trial.json'))
cmrc['dev'] = pd.read_json(os.path.join(data_dir, 'cmrc2018_dev.json'))

In [2]:
cmrc['test']

,context_id,context_text,qas,title
0,TRIAL_800,基于《跑跑卡丁车》与《泡泡堂》上所开发的游戏，由韩国Nexon开发与发行。中国大陆由盛大游戏...,"[{'query_text': '生命数耗完即算为什么？', 'query_id': 'TR...",泡泡战士
1,TRIAL_154,"尤金袋鼠（""Macropus eugenii""）是袋鼠科中细小的成员，通常都是就袋鼠及有袋类...","[{'query_text': '尤金袋鼠分布在哪些地区？', 'query_id': 'T...",尤金袋鼠
2,TRIAL_598,松平康忠（1546年－1618年9月28日）是日本战国时代至安土桃山时代的武将。长泽松平家第...,"[{'query_text': '松平康忠是什么时期的武将？', 'query_id': '...",松平康忠
3,TRIAL_662,乍都节公园（）位于泰国的首都曼谷的乍都节县，是拍凤裕庭路、威拍哇丽兰室路、甘烹碧路之间的一处...,"[{'query_text': '乍都节公园位于什么地方？', 'query_id': 'T...",乍都节公园
4,TRIAL_863,让-皮埃尔·巴利冈（Jean-Pierre Balligand，1950年5月30日 - ）...,"[{'query_text': '让-皮埃尔·巴利冈是哪国的政治家和社会党议员？', 'qu...",让-皮埃尔·巴利冈
...,...,...,...,...
251,TRIAL_311,张天霖（）台湾男演员，毕业于漳和国中、台北市立建国中学补校、真理大学运动管理学系。主要演艺工...,"[{'query_text': '张天霖的著名影视作品有哪些？', 'query_id': ...",张天霖
252,TRIAL_690,广州北回归线标志塔位于中国广东省从化区太平场油麻埔村三甲子一块海拔25.6米的坡地上，是中国...,"[{'query_text': '塔顶的经纬度是多少？', 'query_id': 'TRI...",广州北回归线标志塔
253,TRIAL_697,小榄精神病治疗中心（Siu Lam Psychiatric Centre，原称小榄精神科监狱...,"[{'query_text': '小榄精神病治疗中心属于什么设施？', 'query_id'...",小榄精神病治疗中心
254,TRIAL_141,在数学中，约化群是幂单根为平凡群的代数群。代数环面与半单代数群都是约化群，一般线性群form...,"[{'query_text': '什么是约化群？', 'query_id': 'TRIAL_...",约化群


In [39]:
train_data = []
for _, row in cmrc['train'].iterrows():
    train_data.append({
        "text": row["context_text"],
        "title": row["title"],
    })

In [40]:
train_data

[{'text': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
  'title': '范廷颂'},
 {'text': '安雅·罗素法（，），来自俄罗斯圣彼得堡的模特儿。她是《全美超级模特儿新秀大赛》第十季的亚军。2008年，安雅宣布改回出生时的名字：安雅·罗素法（Anya Rozova），在此之前是使用安雅·冈（）。安雅于俄罗斯出生，后来被一个居住在美国夏威夷群岛欧胡岛檀香山的家庭领养。安雅十七岁时

In [46]:
from config.Config import MilvusConfig
from src.services.vector_db.client import MilvusExecutor
from src.services.vector_db.data_storage import DataDBStorage
from langchain_core.documents import Document

class DataStorage(DataDBStorage):
    def __init__(self,):
        super().__init__()
        # 向量数据库
        self.vector = MilvusExecutor(MilvusConfig(collection_name='cmrc_dataset'))

    async def save_to_vector(self):
        documents = []
        for item in train_data:
            documents.append(Document(page_content=item['text'], title=item['title']))
        await self.vector.client.aadd_documents(documents=documents)


if __name__ == '__main__':
    await DataStorage().save_to_vector()

In [8]:
test_data = []
for _,row in cmrc['test'].iterrows():
    qas = row['qas']
    test_data.append({
        "text": row['context_text'],
        "qas": [{'question':_['query_text'],'answer':"\n".join(_['answers'])} for _ in qas]
    })

In [9]:
test_data

[{'text': '基于《跑跑卡丁车》与《泡泡堂》上所开发的游戏，由韩国Nexon开发与发行。中国大陆由盛大游戏运营，这是Nexon时隔6年再次授予盛大网络其游戏运营权。台湾由游戏橘子运营。玩家以水枪、小枪、锤子或是水炸弹泡封敌人(玩家或NPC)，即为一泡封，将水泡击破为一踢爆。若水泡未在时间内踢爆，则会从水泡中释放或被队友救援(即为一救援)。每次泡封会减少生命数，生命数耗完即算为踢爆。重生者在一定时间内为无敌状态，以踢爆数计分较多者获胜，规则因模式而有差异。以2V2、4V4随机配对的方式，玩家可依胜场数爬牌位(依序为原石、铜牌、银牌、金牌、白金、钻石、大师) ，可选择经典、热血、狙击等模式进行游戏。若游戏中离，则4分钟内不得进行配对(每次中离+4分钟)。开放时间为暑假或寒假期间内不定期开放，8人经典模式随机配对，采计分方式，活动时间内分数越多，终了时可依该名次获得奖励。',
  'qas': [{'question': '生命数耗完即算为什么？', 'answer': '踢爆'},
   {'question': '若游戏中离，则多少分钟内不得进行配对？', 'answer': '4分钟'},
   {'question': '玩家用什么泡封敌人？', 'answer': '玩家以水枪、小枪、锤子或是水炸弹泡封敌人'},
   {'question': '游戏的模式有哪些？', 'answer': '可选择经典、热血、狙击等模式进行游戏。'}]},
 {'text': '尤金袋鼠（"Macropus eugenii"）是袋鼠科中细小的成员，通常都是就袋鼠及有袋类的研究对象。尤金袋鼠分布在澳洲南部岛屿及西岸地区。由于牠们每季在袋鼠岛都大量繁殖，破坏了针鼹岛上的生活环境而被认为是害虫。尤金袋鼠最初是于1628年船难的生还者在西澳发现的，是欧洲人最早有纪录的袋鼠发现，且可能是最早发现的澳洲哺乳动物。尤金袋鼠共有三个亚种：尤金袋鼠很细小，约只有8公斤重，适合饲养。尤金袋鼠的奶中有一种物质，称为AGG01，有可能是一种神奇药及青霉素的改良。AGG01是一种蛋白质，在实验中证实比青霉素有效100倍，可以杀死99%的细菌及真菌，如沙门氏菌、普通变型杆菌及金黄色葡萄球菌。',
  'qas': [{'question': '尤金袋鼠分布在哪些地区？', 'answer': '尤

In [12]:
with open('test_row_data.json', 'w') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=4)